# CEDAR Signature Verification — Training Notebook

## Step-by-step setup for first-time Colab users

### 1. Enable GPU (do this FIRST, before running any cell)
- Click **Runtime** → **Change runtime type**
- Under *Hardware accelerator*, select **T4 GPU** (or better if available)
- Click **Save** — the page will reload, that is normal

### 2. Upload your files
- Click the **folder icon** in the left sidebar (Files panel)
- Click the **upload icon** (page with an up-arrow)
- Upload all **six** files from your local project folder:
  - `preprocess.py`, `model.py`, `dataset.py`, `train.py`, `evaluate.py`
  - `data.zip` — your CEDAR dataset archive
- Wait for the upload progress bar to finish before continuing

### 3. Run all cells in order
- Click **Runtime** → **Run all** (or press Ctrl+F9 / Cmd+F9)
- Training takes roughly 10–30 minutes on a T4 GPU
- The FAR table and histogram are displayed inline when evaluation finishes

### 4. Download your results
- The last cell creates **results.zip** and downloads it automatically
- It contains all epoch checkpoints, the training log, and the evaluation plots

> **Note:** Colab sessions disconnect after ~12 hours or when idle.
> Download results.zip before closing the tab — files are lost on disconnect.

In [ ]:
# ── Cell 1: Verify GPU ────────────────────────────────────────────────────
import torch

if torch.cuda.is_available():
    print(f'GPU : {torch.cuda.get_device_name(0)}')
    print(f'CUDA: {torch.version.cuda}')
else:
    raise SystemExit(
        'No GPU detected. Go to Runtime -> Change runtime type -> GPU, then re-run.'
    )

In [ ]:
# ── Cell 2: Install packages ──────────────────────────────────────────────
# torch + torchvision are pre-installed in Colab; opencv needs installing.
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'opencv-python-headless'], check=True)
print('Packages OK')

In [ ]:
# ── Cell 3: Verify uploads and unzip dataset ──────────────────────────────
import os, zipfile, sys
sys.path.insert(0, '.')

for f in ['preprocess.py','model.py','dataset.py','train.py','evaluate.py','data.zip']:
    if not os.path.exists(f):
        raise FileNotFoundError(f'{f} not found — upload it via the Files panel (see Step 2).')
    print(f'Found {f}')

print('\nUnzipping data.zip ...')
with zipfile.ZipFile('data.zip', 'r') as zf:
    zf.extractall('.')

for folder in ('full_org', 'full_forg'):
    if os.path.isdir(folder):
        print(f'  {folder}/  ({len(os.listdir(folder))} files)')
    else:
        print(f'  WARNING: {folder}/ not found — check data.zip structure')

In [ ]:
# ── Cell 4: Build preprocessed image cache ────────────────────────────────
# Runs the binarise/crop/resize pipeline once for all 2 640 images.
# Re-running this cell is safe — existing .npy files are skipped.
from dataset import build_cache
build_cache('.', 'cache')

In [ ]:
# ── Cell 5: Train (30 epochs, triplet all-pairs loss) ────────────────────
# Strategy: layer4 + fc trainable, conv1-layer3 frozen.
#   - fc   LR = 1e-4  (fast convergence of projection head)
#   - layer4 LR = 1e-5  (10x slower — prevents 1-epoch overfitting)
#   - weight_decay = 5e-3  (50x stronger than before — fights writer memorisation)
#   - augmentation ON, margin=0.5, all-pairs triplet loss
# Expected: epoch-1 loss ~0.40-0.50, slow steady decline, NO collapse to <0.01
#
# Override NUM_WORKERS for Colab (Linux forking works; 2 is enough).
import train as _train_mod
_train_mod.NUM_WORKERS = 2

from train import train
train(sanity=False)

In [ ]:
# ── Cell 6: Evaluate best checkpoint on held-out writers 46-55 ───────────
from evaluate import evaluate
evaluate(checkpoint_path='checkpoints/best.pt')

In [ ]:
# ── Cell 7: Display plots inline ──────────────────────────────────────────
from IPython.display import Image, display
import os

for title, path in [
    ('Score distributions (genuine vs forgery)', 'plots/score_distributions.png'),
    ('FAR / FRR curve',                          'plots/far_frr_curve.png'),
    ('Training loss curve',                      'checkpoints/loss_curve.png'),
]:
    if os.path.exists(path):
        print(title)
        display(Image(path))

In [ ]:
# ── Cell 8: Display FAR operating-point table ─────────────────────────────
import torch
import pandas as pd
from IPython.display import display
from evaluate import (
    load_model, build_all_test_pairs, compute_scores,
    compute_eer, compute_operating_points,
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model  = load_model('checkpoints/best.pt', device)

gp, fp = build_all_test_pairs('.')
gs = compute_scores(model, gp, device, 128, 'genuine')
fs = compute_scores(model, fp, device, 128, 'forgery')

eer, eer_thr, thresholds, fars, frrs = compute_eer(gs, fs)
ops = compute_operating_points(thresholds, fars, frrs, [0.005, 0.01, 0.02, 0.05])

print(f'EER: {eer*100:.2f}%  @ threshold {eer_thr:.4f}\n')

rows = [{
    'FAR target':  f"{r['far_target']*100:.1f}%",
    'Threshold':   f"{r['threshold']:.4f}"       if r['threshold']   is not None else 'N/A',
    'Actual FAR':  f"{r['actual_far']*100:.2f}%" if r['actual_far']  is not None else 'N/A',
    'FRR (miss)':  f"{r['frr']*100:.2f}%"        if r['frr']         is not None else 'N/A',
} for r in ops]

display(pd.DataFrame(rows).set_index('FAR target'))

In [ ]:
# ── Cell 9: Zip and download results ──────────────────────────────────────
import zipfile, os
from google.colab import files

with zipfile.ZipFile('results.zip', 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for d in ('checkpoints', 'plots'):
        if not os.path.isdir(d):
            continue
        for fname in os.listdir(d):
            fpath = os.path.join(d, fname)
            if os.path.isfile(fpath):
                zf.write(fpath)
                print(f'  + {fpath}')

print('\nDownloading results.zip ...')
files.download('results.zip')